# 02.8 - Bit Manipulation

## Why bits?
Computers store everything as **binary** (0s and 1s). Operating directly on bits is extremely fast and memory-efficient.

```
13 in binary = 1101  ->  1·8 + 1·4 + 0·2 + 1·1
```

## Operators
| Operator | Name | Example (a = 0b1100 = 12, b = 0b1010 = 10) | Result |
|---|---|---|---|
| `a & b` | AND - 1 if both are 1 | `1100 & 1010` | `1000` (8) |
| `a \| b` | OR - 1 if either is 1 | `1100 \| 1010` | `1110` (14) |
| `a ^ b` | XOR - 1 if different | `1100 ^ 1010` | `0110` (6) |
| `~a` | NOT | `~12` | `-13` |
| `a << k` | left shift = × 2ᵏ | `3 << 2` | `12` |
| `a >> k` | right shift = // 2ᵏ | `12 >> 2` | `3` |

## Where is it used?
| Real system | Use |
|---|---|
| **File permissions** (Linux `chmod 755`) | Read/write/execute flags packed into bits |
| **Networking** | IP subnet masks (`192.168.1.0/24`), checksums |
| **Feature flags & permissions in apps** | Store many booleans in one integer |
| **Cryptography & hashing** (SHA, AES) | XOR, shifts, rotations |
| **Compression, image formats, graphics** | Packing pixels and colour channels (RGBA) |
| **Bloom filters, bitsets** (databases, caches) | Memory-efficient membership tests |
| **Chess engines** | Bitboards: the whole board in a 64-bit integer |
| **Embedded systems / IoT** | Controlling hardware registers |

In [1]:
a, b = 12, 10
print(f"a = {a:04b}, b = {b:04b}")
print(f"a & b = {a & b:04b} ({a & b})")
print(f"a | b = {a | b:04b} ({a | b})")
print(f"a ^ b = {a ^ b:04b} ({a ^ b})")
print(f"3 << 2 = {3 << 2},  12 >> 2 = {12 >> 2}")
print(bin(13), int("1101", 2), (13).bit_count() if hasattr(int, "bit_count") else bin(13).count("1"))

a = 1100, b = 1010
a & b = 1000 (8)
a | b = 1110 (14)
a ^ b = 0110 (6)
3 << 2 = 12,  12 >> 2 = 3
0b1101 13 3


## Tricks every interviewer expects you to know
| Trick | Code | Why |
|---|---|---|
| Is bit k set? | `(n >> k) & 1` | Shift bit k to position 0 |
| Set bit k | `n \| (1 << k)` | |
| Clear bit k | `n & ~(1 << k)` | |
| Toggle bit k | `n ^ (1 << k)` | |
| Is n even? | `n & 1 == 0` | Last bit |
| Remove lowest set bit | `n & (n - 1)` | `1100 & 1011 = 1000` |
| Is n a power of two? | `n > 0 and n & (n - 1) == 0` | Powers of two have one set bit |
| Lowest set bit | `n & -n` | Two's complement |
| XOR properties | `x ^ x = 0`, `x ^ 0 = x`, order doesn't matter | Pairs cancel out |

In [2]:
# Real-world demo: permission flags (like Linux rwx or app roles)
READ, WRITE, EXECUTE, ADMIN = 1 << 0, 1 << 1, 1 << 2, 1 << 3

user = READ | WRITE                  # grant read + write
print("can write?", bool(user & WRITE))
print("can execute?", bool(user & EXECUTE))
user |= EXECUTE                      # grant execute
user &= ~WRITE                       # revoke write
print(f"permissions bits: {user:04b}")

# chmod 754 -> owner rwx (7=111), group r-x (5=101), others r-- (4=100)
for who, digit in zip(["owner", "group", "other"], "754"):
    d = int(digit)
    print(who, "".join(flag if d & bit else "-" for flag, bit in (("r", 4), ("w", 2), ("x", 1))))

# Subnet mask: is an IP inside 192.168.1.0/24?
def ip_to_int(ip):
    a, b, c, d = map(int, ip.split("."))
    return (a << 24) | (b << 16) | (c << 8) | d
mask = ~((1 << (32 - 24)) - 1) & 0xFFFFFFFF
print("same subnet:", ip_to_int("192.168.1.77") & mask == ip_to_int("192.168.1.0") & mask)

can write? True
can execute? False
permissions bits: 0101
owner rwx
group r-x
other r--
same subnet: True


---
## Solved problems

### 1. Single Number - LC #136 (Easy) Every element appears twice except one. Find it in O(n) time, O(1) space.

**Idea:** XOR everything; duplicates cancel out.

In [3]:
def single_number(nums):
    result = 0
    for x in nums:
        result ^= x          # pairs cancel: x ^ x = 0
    return result

assert single_number([2, 2, 1]) == 1
assert single_number([4, 1, 2, 1, 2]) == 4
print("Single Number ")

Single Number 


### 2. Number of 1 Bits - LC #191 (Easy)
Count set bits (Hamming weight).

In [4]:
def hamming_weight(n):
    count = 0
    while n:
        n &= n - 1           # drop the lowest set bit
        count += 1
    return count

assert hamming_weight(11) == 3        # 1011
assert hamming_weight(128) == 1
assert hamming_weight(2147483645) == 30
print("Number of 1 Bits ")

Number of 1 Bits 


### 3. Counting Bits - LC #338 (Easy) - bits + DP
`ans[i]` = number of 1s in `i`, for 0..n, in O(n).

In [5]:
def count_bits(n):
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i >> 1] + (i & 1)      # i = (i // 2) shifted + last bit
    return ans

assert count_bits(5) == [0, 1, 1, 2, 1, 2]
print("Counting Bits ")

Counting Bits 


### 4. Missing Number - LC #268 (Easy)
`nums` has n distinct numbers from 0..n; one is missing.

**Idea:** XOR all indices 0..n with all values - everything cancels except the missing number. (Sum formula `n(n+1)/2 - sum` also works.)

In [6]:
def missing_number(nums):
    result = len(nums)
    for i, x in enumerate(nums):
        result ^= i ^ x          # every number except the missing one appears twice
    return result

assert missing_number([3, 0, 1]) == 2
assert missing_number([9, 6, 4, 2, 3, 5, 7, 0, 1]) == 8
print("Missing Number ")

Missing Number 


### 5. Power of Two - LC #231 (Easy)

In [7]:
def is_power_of_two(n):
    return n > 0 and n & (n - 1) == 0

assert [is_power_of_two(x) for x in [1, 16, 3, 0, -8, 1024]] == [True, True, False, False, False, True]
print("Power of Two ")

Power of Two 


### 6. Reverse Bits - LC #190 (Easy)
Reverse the bits of a 32-bit unsigned integer.

In [8]:
def reverse_bits(n):
    result = 0
    for _ in range(32):
        result = (result << 1) | (n & 1)    # take the last bit of n, append it to result
        n >>= 1
    return result

assert reverse_bits(0b00000010100101000001111010011100) == 964176192
assert reverse_bits(0b11111111111111111111111111111101) == 3221225471
print("Reverse Bits ")

Reverse Bits 


### 7. Subsets using bitmasks - LC #78 (Medium)
Each number from 0 to 2ⁿ−1 is a subset: bit i set → include `nums[i]`.

In [9]:
def subsets_bitmask(nums):
    n = len(nums)
    return [[nums[i] for i in range(n) if mask >> i & 1] for mask in range(1 << n)]

print(subsets_bitmask([1, 2, 3]))
assert len(subsets_bitmask([1, 2, 3, 4])) == 16
print("Subsets (bitmask) ")

[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]
Subsets (bitmask) 


### 8. Sum of Two Integers - LC #371 (Medium)
Add without `+` or `-` (how CPUs add in hardware).

**Idea:** XOR adds without carrying; AND + shift computes the carry. Repeat until there's no carry.

In [10]:
def get_sum(a, b):
    MASK = 0xFFFFFFFF                    # Python ints are unbounded: simulate 32 bits
    while b & MASK:
        carry = (a & b) << 1             # where both bits are 1 -> carry
        a = a ^ b                        # sum without carry
        b = carry
    a &= MASK
    return a if a <= 0x7FFFFFFF else ~(a ^ MASK)    # back to a negative Python int if needed

assert get_sum(1, 2) == 3 and get_sum(2, 3) == 5 and get_sum(-2, 3) == 1 and get_sum(-5, -7) == -12
print("Sum of Two Integers ")

Sum of Two Integers 


## More practice on LeetCode
Easy: Hamming Distance (#461), Complement of Base 10 Integer (#1009)
Medium: Single Number II (#137), Single Number III (#260), Bitwise AND of Numbers Range (#201), Maximum XOR of Two Numbers (#421)

## Key takeaways
- XOR cancels pairs → "find the odd one out" problems.
- `n & (n - 1)` drops the lowest set bit → counting bits, power-of-two checks.
- A bitmask of n bits represents a subset of n items → useful in DP over subsets.
- Python ints are unbounded: mask with `0xFFFFFFFF` when a problem assumes 32-bit integers.

**Next: `03-Interview Patterns/03.1-Most-Asked-Problems-by-Pattern.ipynb`**